# Day 6 Sample Notebook: Patient-Level Prediction (HADES concepts)

**OHDSI / OMOP Train-the-Trainer, ALS Therapy Development Institute**

Companion to the Day 6 module. This notebook teaches the **shape** of a HADES patient-level prediction
workflow (cohort, covariates, train/test, discrimination, calibration) using scikit-learn on a synthetic
ALS CDM. Production OHDSI prediction is done in R with the HADES `PatientLevelPrediction` package; the
concepts here map directly onto it.

> **Site specific.** This notebook builds a synthetic CDM in memory so it runs in Colab with no credentials and no PHI. To use real data, replace the SQLite connection with your site's governed CDM connection (Databricks, Snowflake, Postgres, BigQuery, and so on). The SQL logic is the same; only the connection changes. Not everyone uses Databricks.

### Objectives
1. Define a prediction problem: at ALS diagnosis, predict gastrostomy within 1 year.
2. Build a covariate table from the CDM.
3. Split into train and test, fit a model, and read the AUC.
4. Check calibration and discuss limitations.

## Step 0: Build the synthetic CDM

In [ ]:
import sqlite3, numpy as np, pandas as pd
np.random.seed(7)

def build_als_cdm(n=250):
    """Build a tiny SYNTHETIC ALS-flavored OMOP CDM in memory. No real data, no PHI."""
    con = sqlite3.connect(":memory:"); cur = con.cursor()
    cur.executescript("""
    CREATE TABLE concept(concept_id INT, concept_name TEXT, domain_id TEXT, vocabulary_id TEXT,
        concept_class_id TEXT, standard_concept TEXT, concept_code TEXT);
    CREATE TABLE concept_ancestor(ancestor_concept_id INT, descendant_concept_id INT, min_levels_of_separation INT);
    CREATE TABLE person(person_id INT, gender_concept_id INT, year_of_birth INT);
    CREATE TABLE observation_period(person_id INT, observation_period_start_date TEXT, observation_period_end_date TEXT);
    CREATE TABLE condition_occurrence(person_id INT, condition_concept_id INT, condition_start_date TEXT);
    CREATE TABLE drug_exposure(person_id INT, drug_concept_id INT, drug_exposure_start_date TEXT);
    CREATE TABLE measurement(person_id INT, measurement_concept_id INT, measurement_date TEXT, value_as_number REAL);
    CREATE TABLE procedure_occurrence(person_id INT, procedure_concept_id INT, procedure_date TEXT);
    """)
    concepts=[
     (35748069,"Amyotrophic lateral sclerosis","Condition","SNOMED","Clinical Finding","S","86044005"),
     (4134454,"Bulbar onset","Observation","SNOMED","Clinical Finding","S","230258005"),
     (1314865,"riluzole","Drug","RxNorm","Ingredient","S","9468"),
     (19077572,"riluzole 50 MG Oral Tablet","Drug","RxNorm","Clinical Drug","S","316158"),
     (1326727,"edaravone","Drug","RxNorm","Ingredient","S","1546020"),
     (40234834,"edaravone 30 MG/100mL","Drug","RxNorm","Clinical Drug","S","1546022"),
     (9990001,"sodium phenylbutyrate / taurursodiol","Drug","RxNorm","Ingredient","S","2630918"),
     (9990002,"tofersen","Drug","RxNorm","Ingredient","S","2629000"),
     (1304643,"baclofen","Drug","RxNorm","Ingredient","S","1292"),
     (4052536,"Gastrostomy","Procedure","SNOMED","Procedure","S","36245001"),
     (3024171,"Forced vital capacity","Measurement","LOINC","Clinical Obs","S","19868-9"),
     (8532,"FEMALE","Gender","Gender","Gender","S","F"),(8507,"MALE","Gender","Gender","Gender","S","M"),
     (0,"No matching concept","Metadata","None","Undefined",None,"0")]
    cur.executemany("INSERT INTO concept VALUES(?,?,?,?,?,?,?)",concepts)
    cur.executemany("INSERT INTO concept_ancestor VALUES(?,?,?)",[
     (1314865,19077572,1),(1314865,1314865,0),(1326727,40234834,1),(1326727,1326727,0),
     (9990001,9990001,0),(9990002,9990002,0),(1304643,1304643,0)])
    persons=[];obsp=[];cond=[];drug=[];meas=[];proc=[]
    for pid in range(1,n+1):
        sex=int(np.random.choice([8532,8507])); yob=int(np.random.normal(1958,11))
        persons.append((pid,sex,yob))
        dx_year=np.random.randint(2018,2024); dx=f"{dx_year}-{np.random.randint(1,13):02d}-15"
        cond.append((pid,35748069,dx))
        bulbar=np.random.rand()<0.3
        if bulbar: cond.append((pid,4134454,dx))
        prior=int(np.random.choice([400,500,800,200,100],p=[.4,.2,.2,.1,.1]))
        start=pd.Timestamp(dx)-pd.Timedelta(days=prior)
        end=pd.Timestamp(dx)+pd.Timedelta(days=int(np.random.randint(365,1500)))
        obsp.append((pid,start.date().isoformat(),end.date().isoformat()))
        base_fvc=float(np.clip(np.random.normal(85-(10 if bulbar else 0),12),30,110))
        r=np.random.rand()
        if r<0.08: pass                                  # missing (completeness)
        elif r<0.11: meas.append((pid,3024171,dx,float(np.random.choice([0,250.0]))))  # implausible (plausibility)
        else: meas.append((pid,3024171,dx,round(base_fvc,1)))
        idx=pd.Timestamp(dx)+pd.Timedelta(days=int(np.random.randint(0,40)))
        if np.random.rand()<0.85:
            drug.append((pid,19077572,idx.date().isoformat()))
            if np.random.rand()<0.45:
                drug.append((pid,40234834,(idx+pd.Timedelta(days=int(np.random.randint(60,300)))).date().isoformat()))
        else:
            drug.append((pid,40234834,idx.date().isoformat()))
        if np.random.rand()<0.15:
            drug.append((pid,9990001,(idx+pd.Timedelta(days=int(np.random.randint(100,500)))).date().isoformat()))
        if bulbar and np.random.rand()<0.4: drug.append((pid,1304643,idx.date().isoformat()))
        if np.random.rand()<0.05: drug.append((pid,0,idx.date().isoformat()))   # unmapped (conformance)
        age=dx_year-yob
        logit=-2.0+0.04*(age-60)+1.1*bulbar-0.03*(base_fvc-80)
        if np.random.rand()<1/(1+np.exp(-logit)):
            proc.append((pid,4052536,(pd.Timestamp(dx)+pd.Timedelta(days=int(np.random.randint(60,360)))).date().isoformat()))
    cur.executemany("INSERT INTO person VALUES(?,?,?)",persons)
    cur.executemany("INSERT INTO observation_period VALUES(?,?,?)",obsp)
    cur.executemany("INSERT INTO condition_occurrence VALUES(?,?,?)",cond)
    cur.executemany("INSERT INTO drug_exposure VALUES(?,?,?)",drug)
    cur.executemany("INSERT INTO measurement VALUES(?,?,?,?)",meas)
    cur.executemany("INSERT INTO procedure_occurrence VALUES(?,?,?)",proc)
    con.commit(); return con

con = build_als_cdm()
def q(sql): return pd.read_sql_query(sql, con)
print("Synthetic ALS CDM ready:", q("SELECT COUNT(*) n FROM person").n[0], "persons")

## Step 1: Define the problem and build covariates
Target cohort: people with an ALS diagnosis. Outcome: a gastrostomy procedure within 365 days of
diagnosis. Covariates: age at diagnosis, sex, bulbar onset, baseline FVC.

In [ ]:
import pandas as pd
df = q("""
SELECT p.person_id,
   (CASE WHEN p.gender_concept_id=8507 THEN 1 ELSE 0 END) AS male,
   co.condition_start_date AS dx, p.year_of_birth AS yob,
   (SELECT 1 FROM condition_occurrence c2 WHERE c2.person_id=p.person_id AND c2.condition_concept_id=4134454) AS bulbar,
   (SELECT value_as_number FROM measurement m WHERE m.person_id=p.person_id AND m.measurement_concept_id=3024171) AS fvc,
   (SELECT pr.procedure_date FROM procedure_occurrence pr WHERE pr.person_id=p.person_id AND pr.procedure_concept_id=4052536) AS gastro_date
FROM person p
JOIN condition_occurrence co ON co.person_id=p.person_id AND co.condition_concept_id=35748069
""")
df["bulbar"] = df["bulbar"].fillna(0)
df["age"] = pd.to_datetime(df.dx).dt.year - df.yob
# outcome: gastrostomy within 365 days of dx
within = (pd.to_datetime(df.gastro_date) - pd.to_datetime(df.dx)).dt.days
df["outcome"] = ((within >= 0) & (within <= 365)).fillna(False).astype(int)
# impute / clean FVC (median; fix implausible)
med = df.loc[(df.fvc>0)&(df.fvc<=120),"fvc"].median()
df["fvc"] = df["fvc"].fillna(med)
df.loc[(df.fvc<=0)|(df.fvc>120),"fvc"] = med
print("rows:", len(df), "| outcome rate:", round(df.outcome.mean(),3))
df[["male","bulbar","fvc","age","outcome"]].head()

## Step 2: Train / test split and fit

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = df[["male","bulbar","fvc","age"]]; y = df["outcome"]
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.3,random_state=1,stratify=y)
model = LogisticRegression(max_iter=1000).fit(Xtr,ytr)
pred = model.predict_proba(Xte)[:,1]
print("Test AUC:", round(roc_auc_score(yte,pred),3))
print("\nCoefficients (log-odds):")
for name,coef in zip(X.columns, model.coef_[0]):
    print(f"  {name:8s} {coef:+.3f}")

## Step 3: Calibration by risk decile
Discrimination (AUC) is not enough. Calibration asks whether predicted risk matches observed risk.

In [ ]:
cal = pd.DataFrame({"pred":pred,"obs":yte.values})
cal["decile"] = pd.qcut(cal["pred"], 5, labels=False, duplicates="drop")
print(cal.groupby("decile").agg(mean_predicted=("pred","mean"),
                                observed_rate=("obs","mean"),
                                n=("obs","size")).round(3))

## Interpretation and the HADES mapping
- In HADES, the **target** and **outcome** cohorts come from ATLAS, covariates are built by
  `FeatureExtraction`, and the model and evaluation come from `PatientLevelPrediction`, which produces
  the same AUC and calibration outputs you see here.
- A model is only as good as the cohort and covariate definitions behind it. Garbage cohorts give
  confident, useless predictions.
- This is synthetic data with a planted signal, so the AUC is optimistic. Real external validation on
  another site's CDM is the OHDSI standard.

## Your turn
1. Drop `bulbar` from the covariates and re-fit. How much does AUC change, and what does that tell you about that feature?
2. Add a covariate for early edaravone use (exposed within 90 days of diagnosis) and re-fit.
3. Why is a held-out test set the minimum, and why does OHDSI push for external validation beyond it?

<details><summary>Answer key</summary>

1. Removing `bulbar` usually lowers AUC, since bulbar onset carries real signal for gastrostomy risk.
2. Build a 0/1 feature by joining edaravone exposures (ingredient 1326727) within 90 days of dx, add to X.
3. A test split guards against in-sample optimism, but a single site can still encode local artifacts.
   External validation on a different CDM tests whether the model transports, which is the real goal.
</details>